In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

os.makedirs('../docs/ml', exist_ok=True)
print("Библиотеки загружены. Директория docs/ml/ готова.")

Библиотеки загружены. Директория docs/ml/ готова.


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

print("--- Часть 0: Исправление Data Leakage ---")

x_raw = pd.DataFrame({"x": [1, 2, 3, 4, 5, 6]})
y_raw = pd.Series([0, 0, 0, 1, 1, 1], name="y_target")

x_train, x_test, y_train, y_test = train_test_split(
    x_raw, y_raw, test_size=0.3, random_state=42, stratify=y_raw
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

pipeline.fit(x_train, y_train)
print("Пайплайн успешно обучен без утечки данных. Тестовый скор:", pipeline.score(x_test, y_test))

--- Часть 0: Исправление Data Leakage ---
Пайплайн успешно обучен без утечки данных. Тестовый скор: 1.0


In [5]:
mart_path = "../data/mart/weather_london_daily_mart.csv"

if os.path.exists(mart_path):
    df = pd.read_csv(mart_path)
    print(f"Данные успешно загружены из локального кэша: {mart_path}")
else:
    print(f"Файл {mart_path} не найден. Генерируем синтетический датасет Лондона за 3 года...")
    date_range = pd.date_range(start="2023-01-01", end="2025-12-31", freq="D")
    np.random.seed(42)
    
    # Исправление: приводим dayofyear к массиву numpy
    day_of_year = date_range.dayofyear.values
    
    base_temp = 10 + 10 * np.sin(2 * np.pi * day_of_year / 365)
    max_temp = base_temp + np.random.normal(3, 2, len(date_range))
    total_precip = np.random.exponential(2, len(date_range))
    
    # Теперь max_temp и total_precip — это обычные массивы numpy, их можно менять
    max_temp[150] = 39.1  
    total_precip[400] = 45.0  
    
    df = pd.DataFrame({
        "date": date_range,
        "city_id": "GB_LON",
        "max_temp": max_temp,
        "total_precip": total_precip
    })

df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month
print(f"Размер датасета: {df.shape}")
print(df.head())

Файл ../data/mart/weather_london_daily_mart.csv не найден. Генерируем синтетический датасет Лондона за 3 года...
Размер датасета: (1096, 5)
        date city_id   max_temp  total_precip  month
0 2023-01-01  GB_LON  14.165562      0.760051      1
1 2023-01-02  GB_LON  13.067688      1.415445      1
2 2023-01-03  GB_LON  14.811574      0.084923      1
3 2023-01-04  GB_LON  16.734084      0.321143      1
4 2023-01-05  GB_LON  13.391341      8.629502      1


In [6]:
print("--- Часть 1: Обучение модели Isolation Forest ---")
features = ["max_temp", "total_precip"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

model = IsolationForest(contamination=0.02, random_state=42)
df["anomaly_label"] = model.fit_predict(X_scaled)
df["anomaly_score"] = model.decision_function(X_scaled)
df["is_anomaly"] = df["anomaly_label"].apply(lambda x: 1 if x == -1 else 0)

anomalies_count = df["is_anomaly"].sum()
print(f"Выявлено аномалий: {anomalies_count} из {len(df)} дней ({anomalies_count/len(df)*100:.2f}%)")

--- Часть 1: Обучение модели Isolation Forest ---
Выявлено аномалий: 22 из 1096 дней (2.01%)


In [7]:
top_anomalies = df[df["is_anomaly"] == 1].sort_values(by="anomaly_score").head(10)
anomalies_report = top_anomalies([["date", "city_id", "max_temp", "total_precip", "anomaly_score"]])

output_csv_path = "../docs/ml/anomalies_top.csv"
anomalies_report.to_csv(output_csv_path, index=False)
print(f"Топ-10 аномалий сохранены в {output_csv_path}")

TypeError: 'DataFrame' object is not callable

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 7))
ax1.plot(df["date"], df["max_temp"], color="steelblue", alpha=0.6, label="Max Temp (°C)")
ax1.set_xlabel("Дата")
ax1.set_ylabel("Температура (°C)", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")

anomalies_df = df[df["is_anomaly"] == 1]
ax1.scatter(anomalies_df["date"], anomalies_df["max_temp"], color="crimson", s=50, label="Выявленная аномалия", zorder=5)

ax2 = ax1.twinx()
ax2.bar(df["date"], df["total_precip"], color="purple", alpha=0.2, label="Precipitation (mm)", width=1)
ax2.set_ylabel("Осадки (мм)", color="purple")
ax2.tick_params(axis="y", labelcolor="purple")

plt.title("Детекция погодных аномалий в Лондоне (Isolation Forest)", fontsize=14, fontweight="bold")
fig.tight_layout()

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

output_img_path = "../docs/ml/metrics.png"
plt.savefig(output_img_path, dpi=200)
plt.close()
print(f"График сохранен в {output_img_path}")

In [ ]:
summary_text = f"""# Отчет по внедрению ML-слоя (Аномалии) — Неделя 13

## 1. Защита от Data Leakage (Часть 0)
* Масштабирование признаков выполняется строго внутри изолированного пайплайна.
* Поиск аномалий производится без использования целевых меток (Unsupervised Learning).

## 2. Постановка задачи и Методология
* **Задача:** Выявление экстремальных климатических аномалий в Лондоне.
* **Признаки:** max_temp и total_precip.
* **Модель:** sklearn.ensemble.IsolationForest.

## 3. Результаты анализа
* Всего обработано дней: {len(df)}
* Обнаружено аномальных дней: {anomalies_count}

## 4. Вывод о практической полезности
ML-блок позволяет автоматически отлавливать климатические рекорды и технические сбои API датчиков.
"""
summary_path = "../docs/ml/week13_summary.md"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_text)
print(f"Финальный отчет сгенерирован в {summary_path}")